# QLoRA Fine-Tuning — **Qwen 2.5 14B** Instruct on the Egyptian Civil Code knowledge dataset (Unsloth)

This is **Experiment 5 (Stage A) scaled to 14B** — the biggest single quality lever available for this task. Same dataset (`data/qa_pairs_knowledge.jsonl` — 22,582 examples, 10 task families, 100 % of 1,093 articles, ~20 examples/article), same memorisation-shaped recipe, on a much bigger base.

| Why | What |
|---|---|
| **Base** | `Qwen/Qwen2.5-14B-Instruct` — the strongest balanced EN + AR open instruct model that's still trainable on a single GPU. Best Arabic *and* English at the 14B tier; same family as the 1.5B / 3B local runs (clean scaling story 1.5B → 3B → 7B → 14B). |
| **Engine** | **Unsloth** — required at 14B in 4-bit; vanilla `transformers` + PEFT won't fit a 14B QLoRA with seq ≥ 1024 even on an A100 24 GB. Unsloth cuts ~50 % of activation VRAM. |
| **Recipe** | r=32 / α=64 / dropout=0 / **LR 3e-5** / 4 epochs (memorisation shape; LR gentler than the 7B's 5e-5 because larger models are more parameter-sensitive) |
| **Compute target** | **T4 16 GB: NOT viable** (4-bit base alone ≈ 8 GB, no activation headroom). **L4 24 GB: tight** — drop `MAX_SEQ_LEN` to 1280, watch peak VRAM. **A100 40 GB: recommended**, comfortable headroom for seq 1536. |
| **Wall-clock** | ~12-18 h on A100 40 GB; ~18-24 h on L4 24 GB; impractical on T4 |

> If A100 isn't available and L4 is tight, the **7B notebook** ([`qlora_qwen2_5_7b_knowledge_colab.ipynb`](./qlora_qwen2_5_7b_knowledge_colab.ipynb)) is the right fallback — fits T4, much faster, ~80 % of the eventual quality.

**Want to test an alternative Arabic-strong base?** Change `BASE_MODEL` in Step 2 — shortlist in that cell's comment.

## Step 1 — Environment & GPU check

In [ ]:
!nvidia-smi
import torch, sys
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  VRAM={p.total_memory/1e9:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")
    if p.total_memory/1e9 < 22:
        print("⚠️  14B QLoRA is risky on <24 GB. Consider the 7B notebook instead, "
              "or drop MAX_SEQ_LEN to 1024 in Step 2.")
print("python:", sys.version.split()[0])

In [ ]:
# Pinned, mutually-compatible stack: Unsloth pulls a known-good torch/triton.
# (Run once per Colab session; ~3 min.)
%pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps "trl<0.10" "peft>=0.13" "accelerate>=0.32" "bitsandbytes>=0.43"
%pip install -q "transformers>=4.45,<5" "datasets>=2.21" "tensorboard" "matplotlib" "pyyaml"
print('Stack installed. Restart runtime only if Colab nags about it.')

## Step 2 — Configuration

`BASE_MODEL` is the only thing you'd swap to try a different base — see the comment for shortlisted options.

In [ ]:
# ============================================================
# >>> CHOOSE YOUR BASE MODEL HERE <<<
# Shortlist of Arabic + English-strong instruct models at 14B-class size.
# All Unsloth-compatible.
#
#   Qwen 2.5 14B (recommended — best EN+AR balance at this size; same family as
#   our 1.5B/3B local runs):
#     "unsloth/Qwen2.5-14B-Instruct"           # ~8 GB 4-bit. A100 / L4 (tight).
#
#   Qwen 3 14B (newer line, often stronger reasoning + multilingual):
#     "unsloth/Qwen3-14B"                      # similar size, similar fit.
#
#   Arabic-specialised (use only as a comparison ablation):
#     "inceptionai/jais-family-13b-chat"       # Arabic-first; English noticeably weaker.
#
#   Drop to 7B if VRAM is too tight (see qlora_qwen2_5_7b_knowledge_colab.ipynb).
# ============================================================

BASE_MODEL     = "unsloth/Qwen2.5-14B-Instruct"
ADAPTER_NAME   = "qlora-qwen2.5-14b-knowledge"
OUTPUT_DIR     = f"/content/runs/{ADAPTER_NAME}"

# LoRA — memorisation shape. Same r as the 7B notebook (capacity for 1,093 articles).
LORA_R         = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0          # MUST be 0 for Unsloth to fully patch LoRA layers
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Training
EPOCHS            = 4
PER_DEVICE_BS     = 1
GRAD_ACCUM        = 16            # effective batch = 16
LR                = 3e-5          # gentler than 7B's 5e-5 (14B is more parameter-sensitive)
WARMUP_RATIO      = 0.03
MAX_SEQ_LEN       = 1536          # A100: keep at 1536.  L4: drop to 1280.  T4: not supported.
SEED              = 13

# Eval & checkpointing
EVAL_STRATEGY     = "epoch"
SAVE_STRATEGY     = "epoch"
SAVE_TOTAL_LIMIT  = 2
LOAD_BEST_AT_END  = True
EARLY_STOP_PATIENCE = 5

# Optional Weights & Biases (set WANDB_API_KEY in Colab Secrets to enable; otherwise TensorBoard only)
USE_WANDB         = False
WANDB_PROJECT     = "legalpolicy-knowledge"
WANDB_RUN_NAME    = ADAPTER_NAME

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Base model: {BASE_MODEL}")
print(f"Output:     {OUTPUT_DIR}")
print(f"Recipe:     r={LORA_R}/α={LORA_ALPHA}, LR={LR}, {EPOCHS} epochs, max_seq={MAX_SEQ_LEN}")

## Step 3 — Upload the knowledge dataset

Three ways to get `qa_pairs_knowledge.jsonl` + `qa_pairs_knowledge_val.jsonl` into the runtime. Pick one.

In [ ]:
# ============================================================
# Choose ONE of: 'upload' | 'drive' | 'github' | 'build_inline'
# ============================================================
DATA_SOURCE = 'upload'

TRAIN_PATH = '/content/qa_pairs_knowledge.jsonl'
VAL_PATH   = '/content/qa_pairs_knowledge_val.jsonl'

if DATA_SOURCE == 'upload':
    from google.colab import files
    print('Upload qa_pairs_knowledge.jsonl AND qa_pairs_knowledge_val.jsonl from your repo data/ dir:')
    up = files.upload()
    for name in up:
        if 'val' in name:
            VAL_PATH = '/content/' + name
        else:
            TRAIN_PATH = '/content/' + name

elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    TRAIN_PATH = '/content/drive/MyDrive/LegalPolicy_LLM/data/qa_pairs_knowledge.jsonl'
    VAL_PATH   = '/content/drive/MyDrive/LegalPolicy_LLM/data/qa_pairs_knowledge_val.jsonl'

elif DATA_SOURCE == 'github':
    GH_RAW = 'https://raw.githubusercontent.com/<user>/<repo>/<branch>/data'
    !curl -sSL {GH_RAW}/qa_pairs_knowledge.jsonl     -o {TRAIN_PATH}
    !curl -sSL {GH_RAW}/qa_pairs_knowledge_val.jsonl -o {VAL_PATH}

elif DATA_SOURCE == 'build_inline':
    !curl -sSLO https://raw.githubusercontent.com/<user>/<repo>/<branch>/src/legal_explainer/finetune/knowledge_builder.py
    raise NotImplementedError("Wire this up if you only have orig_data.json on hand.")

import json
with open(TRAIN_PATH) as f: n_train = sum(1 for _ in f)
with open(VAL_PATH)   as f: n_val   = sum(1 for _ in f)
print(f'train: {n_train} examples  →  {TRAIN_PATH}')
print(f'val:   {n_val} examples  →  {VAL_PATH}')

with open(TRAIN_PATH) as f:
    rec = json.loads(f.readline())
print('\nfirst record:', rec.get('kind'), '·', rec.get('language'), '·', rec.get('article_key'))
print('  user:    ', rec['messages'][0]['content'][:160])
print('  assist:  ', rec['messages'][1]['content'][:160])

## Step 4 — Load Qwen 2.5 14B with Unsloth + attach LoRA

Unsloth must be imported **before** torch / transformers so its kernel patches apply.

14B base in 4-bit will take ~30-60 s to download/load.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,           # auto: bf16 on Ampere+, fp16 otherwise
    load_in_4bit    = True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,   # 0 → Unsloth fully patches
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",      # Unsloth's lower-VRAM impl
    random_state               = SEED,
)
model.print_trainable_parameters()

# Quick VRAM probe BEFORE training — if this is already > 80 % full, drop MAX_SEQ_LEN.
free, total = torch.cuda.mem_get_info()
print(f"VRAM after model + LoRA: used={(total-free)/1e9:.2f}/{total/1e9:.1f} GB  "
      f"({100*(total-free)/total:.0f}%)")

## Step 5 — Tokenise the dataset using Qwen's ChatML template

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={'train': TRAIN_PATH, 'validation': VAL_PATH})

def tokenize(ex):
    text = tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)
    return tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN, padding=False)

drop_cols = ds['train'].column_names
ds = ds.map(tokenize, remove_columns=drop_cols, desc='tokenize')
print('train:', len(ds['train']), '·  val:', len(ds['validation']))
print('first record keys:', list(ds['train'][0].keys()))
print('first 40 input_ids:', ds['train'][0]['input_ids'][:40])


## Step 6 — Trainer + experiment tracking

TRL's `SFTTrainer`. TensorBoard always on; W&B if `USE_WANDB=True` and `WANDB_API_KEY` is in Colab Secrets.

In [ ]:
import os
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

report_to = ['tensorboard']
if USE_WANDB:
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except Exception:
        pass
    if os.environ.get('WANDB_API_KEY'):
        import wandb
        wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, job_type='train')
        report_to = ['wandb','tensorboard']
    else:
        print('No WANDB_API_KEY in Colab Secrets — falling back to TensorBoard only.')

sft = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    run_name                    = ADAPTER_NAME,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BS,
    per_device_eval_batch_size  = PER_DEVICE_BS,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = WARMUP_RATIO,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    optim                       = 'paged_adamw_8bit',
    logging_steps               = 10,
    eval_strategy               = EVAL_STRATEGY,
    eval_steps                  = 50,
    save_strategy               = SAVE_STRATEGY,
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = LOAD_BEST_AT_END,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = report_to,
    logging_dir                 = f'{OUTPUT_DIR}/runs',
    max_seq_length                  = MAX_SEQ_LEN,
    dataset_kwargs              = {"skip_prepare_dataset": True},
    packing                     = False,
    seed                        = SEED,
)

trainer = SFTTrainer(
    model            = model,
    args             = sft,
    train_dataset    = ds['train'],
    eval_dataset     = ds['validation'],
    tokenizer        = tokenizer,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)
print('Trainer ready.')

## Step 7 — Train (long-running cell, 12-18 h on A100)

Watch the loss curve in TensorBoard while it runs.

```python
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/runs
```

If Colab disconnects mid-run, restart this notebook, **set `RESUME=True` in the cell below**, and run Steps 1–6 then Step 7 — HF Trainer will pick up from the latest `checkpoint-*` in `OUTPUT_DIR`.

> **A100 / L4 tip:** prefer the Colab background-execution setting (Colab Pro) so disconnects don't kill the kernel.

In [ ]:
# Resume control:
#   RESUME = False  → fresh training run (overwrites OUTPUT_DIR)
#   RESUME = True   → auto-detect latest checkpoint in OUTPUT_DIR
#   RESUME = "/content/runs/<name>/checkpoint-1355"  → specific checkpoint
RESUME = False

trainer.train(resume_from_checkpoint=RESUME if RESUME is not False else None)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved → {OUTPUT_DIR}')

## Step 8 — Closed-book smoke test

Same eyeball test we ran on the 1.5B v1 adapter locally — quote articles verbatim with no article in the prompt, identify articles from snippets. Reference articles mirror the local test set (775, 990, 1068, 280, 836) so the 14B number is directly comparable to the 1.5B v1 (char-sim 0.44, 2/8 exact).

In [ ]:
from difflib import SequenceMatcher
import re, json, time

TEST_ARTICLES = [
    ('Article 775', 'en'),
    ('Article 990', 'ar'),
    ('Article 1068', 'en'),
    ('Article 280', 'en'),
    ('Article 836', 'ar'),
]

# If you uploaded orig_data.json to /content/ this auto-grades; otherwise it just shows generations.
ORIG_PATH = '/content/orig_data.json'
orig = None
if os.path.exists(ORIG_PATH):
    orig = json.load(open(ORIG_PATH))

def normalize(t):
    t = re.sub(r'\s+', ' ', t or '').strip()
    return re.sub(r"[،؛؟.,;:!?\"'()\[\]{}«»\-—–]", ' ', t).strip()

def char_sim(p, g):
    return SequenceMatcher(None, normalize(p).lower(), normalize(g).lower()).ratio()

FastLanguageModel.for_inference(model)
for key, lang in TEST_ARTICLES:
    n = key.split()[1]
    q = (f'Quote {key} of the Egyptian Civil Code exactly as it is written.'
         if lang == 'en' else
         f'اذكر نص المادة {n} من القانون المدني المصري حرفياً.')
    chat = [{'role':'user','content':q}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=600, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    sec = time.perf_counter() - t0
    pred = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f'\n=== {key} ({lang}) — {sec:.1f}s ===')
    if orig and key in orig:
        gold = (orig[key].get('english') if lang=='en' else orig[key].get('arabic')) or ''
        print(f'char-sim vs gold: {char_sim(pred, gold):.2f}')
        print('GOLD:', gold[:300])
    print('PRED:', pred[:600])

## Step 9 — Save the adapter

Three options to persist past the runtime:

- **A) Drive** — copy `OUTPUT_DIR` to your Google Drive
- **B) HuggingFace Hub** — push it (set `HF_TOKEN` in Colab Secrets first)
- **C) Local laptop** — zip and download (the adapter is ~150 MB for 14B r=32)

In [ ]:
# === A) Drive ===
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    !mkdir -p /content/drive/MyDrive/LegalPolicy_LLM/runs
    !cp -r {OUTPUT_DIR} /content/drive/MyDrive/LegalPolicy_LLM/runs/
    print('Saved →', f'/content/drive/MyDrive/LegalPolicy_LLM/runs/{ADAPTER_NAME}')

# === B) HuggingFace Hub ===
PUSH_TO_HUB = False
HUB_REPO = 'your-username/qlora-qwen2.5-14b-knowledge'
if PUSH_TO_HUB:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
    model.push_to_hub(HUB_REPO, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(HUB_REPO, token=os.environ['HF_TOKEN'])
    print('Pushed →', HUB_REPO)

# === C) Zip + download ===
DOWNLOAD_ZIP = False
if DOWNLOAD_ZIP:
    !cd /content/runs && zip -qr {ADAPTER_NAME}.zip {ADAPTER_NAME}
    from google.colab import files
    files.download(f'/content/runs/{ADAPTER_NAME}.zip')

## Step 10 — (Optional) Stage B continuation on top of this adapter

Once Step 8 looks good, the natural next step is **Stage B** — continue training the same adapter on the existing house-style SFT data (`qa_pairs.jsonl` from the local repo) so it learns both *what* the law says (this Stage A) *and* *how* we want it explained.

Easiest path:
1. Push this adapter to HF Hub (Step 9 above)
2. Open a fresh Colab notebook
3. Load this adapter as the starting point (instead of a fresh base)
4. Train on `qa_pairs.jsonl` for 2-3 epochs at LR 1e-5 (gentler — 14B + house style needs much less push than 14B + raw memorisation)

A wrapper script `scripts/train_two_stage.py` is on the project's todo list.